# Semifinal 03 - Hashtag frequency per candidate for word clouds

In [ ]:
DATA_DIR = "../../data/semifinal"

In [ ]:
!pip install -q nlp_id

In [ ]:
import pandas as pd
import numpy as np
import re

## Helper

In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # # fix 3
    # text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9# ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            # new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
            new_text.append(word)
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "maap": "maaf",
        "ga" : "tidak",
        "nggak" : "tidak"
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text.lower()

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

def extract_hashtag(text):
    hashtags = [word.lower() for word in re.findall(r'#(\w+)', text)]
    return list(set(hashtags))

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/sample_with_paslon.csv")
df.head()

             created_at                                       username  \
0  2024-01-19T08:27:10Z  @yyg6obgSqnNzskShET5StvYh80qgordLCI/nDl9sRfU=   
1  2024-01-31T10:12:17Z  @bULgmJclTgk4cEU0y+7RywiDMKTR5zhJ0ikshOSNVIA=   
2  2024-01-25T08:05:32Z  @QqgrJZpi+DbOhagCq/IMLIPF/yyRaImZ1OyPl+hdQEw=   
3  2024-01-01T11:17:08Z  @YRxZlLvPiEOnVprXzbXwKUxeSfqC8peiPEKhagn0Iew=   
4  2024-01-17T16:23:26Z  @UJxsoI0fCW5egl24YbD4qlTyBPGaKcUHnqj8y8T6gzY=   

     tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
0  mention             0  twit     72.0     43.0   1501.0   
1       rt            27  twit    733.0    227.0   3050.0   
2       rt          1976  twit    353.0     53.0    346.0   
3       rt          1379  twit    416.0    943.0  25924.0   
4       rt            56  twit   1196.0   1051.0  14148.0   

                         loc  lst_cnt  \
0                     Sorong      0.0   
1                        NaN      0.0   
2                     Jekate      0.0   
3        Beautiful Indon

In [ ]:
df.shape

(5000, 15)

## Prabowo

In [ ]:
# counter prabowo
prabowo_df = df[df['is_prabowo'] == 1]
prabowo_df.shape

(1725, 15)

In [ ]:
from nlp_id.lemmatizer import Lemmatizer
from nlp_id.stopword import StopWord
stopword = StopWord()
lemmatizer = Lemmatizer()
lemmatizer.lemmatize('Saya sedang mencoba')

'saya sedang coba'

In [ ]:
# prabowo_df['content'] = prabowo_df['content'].apply(remove_noise)
# prabowo_df['content'] = prabowo_df['content'].apply(normalize_word)
# prabowo_df['content_lemmatized'] = prabowo_df['content'].apply(lambda x: lemmatizer.lemmatize(x))
# prabowo_df['content_lemmatized'] = prabowo_df['content'].apply(lambda x: stopword.remove_stopword(x))

In [ ]:
prabowo_df["hashtags"] = prabowo_df["content"].apply(extract_hashtag)
prabowo_df.head()

<ipython-input-55-fcbb1ddbbf40>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prabowo_df["hashtags"] = prabowo_df["content"].apply(extract_hashtag)


              created_at                                       username  \
2   2024-01-25T08:05:32Z  @QqgrJZpi+DbOhagCq/IMLIPF/yyRaImZ1OyPl+hdQEw=   
7   2024-01-21T04:36:13Z  @Ckm3Puh0c4p68jKIozohC2sd4l+kgyVNeLZPKu8zi+A=   
8   2024-01-11T03:33:50Z  @2dZfmlrbQhq2/mO3qqxXrcDPS6STGxoqvHxcq75TIHk=   
10  2024-01-03T11:57:09Z  @Y5U2IQkOoUQgBdxJ/e6MzaKjwY2BQqkcF/vO+bahlP0=   
18  2024-01-16T17:22:19Z  @9dEJQJS62xuhQwF0CUg2HwqEHyYmlZqQQaiovt9lkSw=   

      tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
2        rt          1976  twit    353.0     53.0    346.0   
7        rt             1  twit     21.0      3.0    383.0   
8   mention             0  twit     33.0      0.0    639.0   
10  mention             0  twit     47.0      9.0    345.0   
18    reply             0  twit     36.0      1.0    160.0   

                       loc  lst_cnt  \
2                   Jekate      0.0   
7                      NaN      0.0   
8   Bosnia and Herzegovina      0.0   
10         Pematangs

In [ ]:
stop_tags = [
    "ganjar",
    "anis",
    "imin",
    "anies",
    "mahfud",
    "1",
    "3",
    "amin",
    "jnk"
]

In [ ]:
word_count = dict()
for i in range(len(prabowo_df)):
  for word in prabowo_df.iloc[i]['hashtags']:
    is_not_relevant = False
    for stop_tag in stop_tags:
      if stop_tag in word:
        is_not_relevant = True
        break
    if not is_not_relevant:
      word_count[word] = word_count.get(word, 0) + 1

word_count = pd.DataFrame.from_dict(word_count, orient='index').reset_index()
word_count.columns = ['hashtag', 'count']
word_count = word_count.sort_values(by='count', ascending=False)
word_count.reset_index(drop=True, inplace=True)
word_count.head()

                 hashtag  count
0          02melanjutkan     73
1  anakmudaindonesiaemas     42
2       indonesiasentris     42
3         indonesiahijau     42
4       asalbukanprabowo     39

In [ ]:
word_count

                   hashtag  count
0            02melanjutkan     73
1    anakmudaindonesiaemas     42
2         indonesiasentris     42
3           indonesiahijau     42
4         asalbukanprabowo     39
..                     ...    ...
330          indonesiaraya      1
331                 rakyat      1
332                 pangan      1
333              pertanian      1
334              editorial      1

[335 rows x 2 columns]

In [ ]:
word_count["count_scaled"] = word_count["count"] / 146 * 100
word_count["count_scaled"] = word_count["count_scaled"] + 20
word_count.head()

                 hashtag  count  count_scaled
0          02melanjutkan     73     50.000000
1  anakmudaindonesiaemas     42     28.767123
2       indonesiasentris     42     28.767123
3         indonesiahijau     42     28.767123
4       asalbukanprabowo     39     26.712329

In [ ]:
word_count.to_csv(f"{DATA_DIR}/prabowo_hashtag_count.csv", index=False)

In [ ]:
prabowo_df[prabowo_df.content.str.lower().str.contains("jnk")]

                created_at                                       username  \
794   2024-01-22T19:27:07Z  @C3h0ZigBZ3a1AFXma6gN3C6Ja1SmcLWRmkRxkma5o4E=   
1360  2024-01-17T12:40:11Z  @yQvlnEzQ9rA/z/sKvVISm7uCbZp+IzAitLZ+J7Zvwoc=   
1557  2024-01-19T11:05:14Z  @VwbeTwwYoT4sJ0voAU/EcBQAqtoNQNSPL0Hkot/tgQg=   
2034  2024-01-17T12:05:13Z  @d8vm6vv+C9HqeqzYmkstIsPQZMgJAJG0+9nhLMiRYbs=   
2337  2024-01-19T06:50:12Z  @PPP+4dAE9K4vyN25GlVcghMciFi6HIeaejiJVX6M75c=   
3016  2024-01-17T09:50:12Z  @rNCGI1ZXmazNj+Bk4zoJjTIoZdTH2z6+DDFV/OnSZKU=   
3392  2024-01-19T03:53:14Z  @VVx4IjFSOCH7GDeZPmX3YCePXfp0P6A9OvG46293x0Y=   
3924  2024-01-18T17:28:34Z  @y8TgdXNaEy/l6zgW3YtULX3AgdHbv98ufdq+hoKcBP4=   
4305  2024-01-15T19:35:11Z  @bsKFfsH28lI+pbAxL8M9oUx4UlyHmZkSuh4HaqqlcsI=   
4317  2024-01-11T23:20:12Z  @ZKIEll1YBFyL90+MWhe+CJmlnlUimK9Ir0pKaj6U6bA=   
4496  2024-01-18T17:23:35Z  @m/a3A3k8vJZYjB21lP+Sl0nCpD2nvLWY8WOefLTGmrM=   

        tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt            loc

## Anies

In [ ]:
# counter anies
anies_df = df[df['is_anies'] == 1]
anies_df.shape

(2406, 15)

In [ ]:
anies_df['content'] = anies_df['content'].apply(remove_noise)
anies_df['content'] = anies_df['content'].apply(normalize_word)

<ipython-input-105-5a61f0142b76>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anies_df['content'] = anies_df['content'].apply(remove_noise)
<ipython-input-105-5a61f0142b76>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anies_df['content'] = anies_df['content'].apply(normalize_word)


In [ ]:
anies_df["hashtags"] = anies_df["content"].apply(extract_hashtag)
anies_df.head()

<ipython-input-106-28c41fe7fba7>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anies_df["hashtags"] = anies_df["content"].apply(extract_hashtag)


              created_at                                       username  \
1   2024-01-31T10:12:17Z  @bULgmJclTgk4cEU0y+7RywiDMKTR5zhJ0ikshOSNVIA=   
3   2024-01-01T11:17:08Z  @YRxZlLvPiEOnVprXzbXwKUxeSfqC8peiPEKhagn0Iew=   
6   2024-01-13T13:06:17Z  @qgVFW6qHKBk5799XdpO0pjfiSQVpGxy0IINo2VpGAYY=   
10  2024-01-03T11:57:09Z  @Y5U2IQkOoUQgBdxJ/e6MzaKjwY2BQqkcF/vO+bahlP0=   
11  2024-01-27T05:38:10Z  @cpDeEC4CpazoWsaXMVMw261PSkO1trlImfku18LOerY=   

      tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
1        rt            27  twit    733.0    227.0   3050.0   
3        rt          1379  twit    416.0    943.0  25924.0   
6        rt           399  twit    493.0    286.0  52372.0   
10  mention             0  twit     47.0      9.0    345.0   
11       rt           979  twit    200.0     18.0   6833.0   

                          loc  lst_cnt  \
1                         NaN      0.0   
3         Beautiful Indonesia      1.0   
6   Tarakan, Kalimantan utara      0.0   
10      

In [ ]:
# show rows where hashtags have "text". note hashtags is a list
anies_df.loc[anies_df["content"].str.contains("#text"), "content"].values

array(['anies pilih atasi stunting pakai ikan bukan susu mudah didapat harga murah kata anies kalau pakai susu cuma pabrik susu yang hidup karena proyek itu #newsupdate #update #news #text kumparan',
       'csis prabowo totally lost hanya fokus pukul balik anies #newsupdate #update #news #text kumparan',
       'bagi saya partai politik adalah salah satu partai yang konsisten dalam menjaga demokrasi dan ibu mega penjaga demokrasi yang konsisten berani kata anies #newsupdate #update #news #text kumparan',
       'sebanyak 70 73 persen pembaca kumparan pilih anies cak imin usai debat ketiga pilpres angka ini merupakan hasil polling kumparan #newsupdate #news #update #text kumparan',
       'anies saya terkejut pak presiden kok berkomentar soal debat ya #newsupdate #update #news #text kumparan',
       'anies ucapkan terima kasih untuk pembuat akun anies bubble di x #newsupdate #update #news #text kumparan',
       'belum lama dipasang videotron yang bikin anies terharu diturunkan #newsu

In [ ]:
stop_tags = [
    "ganjar",
    "prabowo",
    "gibran",
    "mahfud",
    "2",
    "3",
    "jnk",
    "gemoy",
    "samsul"
]

In [ ]:
word_count = dict()
for i in range(len(anies_df)):
  for word in anies_df.iloc[i]['hashtags']:
    is_not_relevant = False
    for stop_tag in stop_tags:
      if stop_tag in word:
        is_not_relevant = True
        break
    if not is_not_relevant:
      word_count[word] = word_count.get(word, 0) + 1

word_count = pd.DataFrame.from_dict(word_count, orient='index').reset_index()
word_count.columns = ['hashtag', 'count']
word_count = word_count.sort_values(by='count', ascending=False)
word_count.reset_index(drop=True, inplace=True)
word_count.head()

                      hashtag  count
0                 aminajadulu    216
1                       anies     60
2                  desakanies     53
3                 debatcapres     45
4  pahitmaniesalwayswithanies     36

In [ ]:
word_count

                        hashtag  count
0                   aminajadulu    216
1                         anies     60
2                    desakanies     53
3                   debatcapres     45
4    pahitmaniesalwayswithanies     36
..                          ...    ...
256                       ambon      1
257                      maluku      1
258                     abahowl      1
259                coblosnomor1      1
260                aniesdiancam      1

[261 rows x 2 columns]

In [ ]:
word_count["count_scaled"] = word_count["count"] / 432 * 100
word_count["count_scaled"] = word_count["count_scaled"] + 20
word_count.head()

                      hashtag  count  count_scaled
0                 aminajadulu    216     70.000000
1                       anies     60     33.888889
2                  desakanies     53     32.268519
3                 debatcapres     45     30.416667
4  pahitmaniesalwayswithanies     36     28.333333

In [ ]:
word_count.tail()

          hashtag  count  count_scaled
256         ambon      1     20.231481
257        maluku      1     20.231481
258       abahowl      1     20.231481
259  coblosnomor1      1     20.231481
260  aniesdiancam      1     20.231481

In [ ]:
word_count.to_csv(f"{DATA_DIR}/anies_hashtag_count.csv", index=False)

## Ganjar

In [138]:
# counter ganjar
ganjar_df = df[df['is_ganjar'] == 1]
ganjar_df.shape

(1418, 15)

In [139]:
# anies_df['content'] = anies_df['content'].apply(remove_noise)
# anies_df['content'] = anies_df['content'].apply(normalize_word)

In [140]:
ganjar_df["hashtags"] = ganjar_df["content"].apply(extract_hashtag)
ganjar_df.head()

<ipython-input-140-96800502e1e4>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ganjar_df["hashtags"] = ganjar_df["content"].apply(extract_hashtag)


             created_at                                       username  \
0  2024-01-19T08:27:10Z  @yyg6obgSqnNzskShET5StvYh80qgordLCI/nDl9sRfU=   
4  2024-01-17T16:23:26Z  @UJxsoI0fCW5egl24YbD4qlTyBPGaKcUHnqj8y8T6gzY=   
5  2024-01-19T22:55:12Z  @dY2cwjOu8tX1S/m14gqu6o+bFgrE2Yir4QEpcWjIkqg=   
8  2024-01-11T03:33:50Z  @2dZfmlrbQhq2/mO3qqxXrcDPS6STGxoqvHxcq75TIHk=   
9  2024-01-17T16:09:37Z  @tlzohb6n/6pappbSz0xuD/7GdoD4N5kzGdyJYIacLvE=   

     tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
0  mention             0  twit     72.0     43.0   1501.0   
4       rt            56  twit   1196.0   1051.0  14148.0   
5  mention             0  twit      2.0      2.0   4840.0   
8  mention             0  twit     33.0      0.0    639.0   
9       rt            15  twit      0.0      0.0    749.0   

                         loc  lst_cnt  \
0                     Sorong      0.0   
4  Kota Surabaya, Jawa Timur      7.0   
5                        NaN      0.0   
8     Bosnia and Herzego

In [141]:
# show rows where hashtags have "text". note hashtags is a list
ganjar_df.loc[ganjar_df["content"].str.lower().str.contains("makansianggratis"), "content"].values

array([], dtype=object)

In [142]:
stop_tags = [
    "anis",
    "prabowo",
    "gibran",
    "anies",
    "2",
    "1",
    "amin",
    "gemoy",
    "samsul",
    "imin",
    "slepet"
]

In [143]:
word_count = dict()
for i in range(len(ganjar_df)):
  for word in ganjar_df.iloc[i]['hashtags']:
    is_not_relevant = False
    for stop_tag in stop_tags:
      if stop_tag in word:
        is_not_relevant = True
        break
    if not is_not_relevant:
      word_count[word] = word_count.get(word, 0) + 1

word_count = pd.DataFrame.from_dict(word_count, orient='index').reset_index()
word_count.columns = ['hashtag', 'count']
word_count = word_count.sort_values(by='count', ascending=False)
word_count.reset_index(drop=True, inplace=True)
word_count.head()

                    hashtag  count
0                       jnk    448
1       ganjarmahfudrebound    423
2  ganjarpranowopilihanumat    401
3                   coblos3    227
4                       3gm    148

In [144]:
word_count

                      hashtag  count
0                         jnk    448
1         ganjarmahfudrebound    423
2    ganjarpranowopilihanumat    401
3                     coblos3    227
4                         3gm    148
..                        ...    ...
208  harirayaganjarianspartan      1
209            indonesiahebat      1
210                      pdip      1
211                  megawati      1
212        indonesialebihbaik      1

[213 rows x 2 columns]

In [145]:
word_count["count_scaled"] = word_count["count"] / 896 * 100
word_count["count_scaled"] = word_count["count_scaled"] + 20
word_count.head()

                    hashtag  count  count_scaled
0                       jnk    448     70.000000
1       ganjarmahfudrebound    423     67.209821
2  ganjarpranowopilihanumat    401     64.754464
3                   coblos3    227     45.334821
4                       3gm    148     36.517857

In [146]:
word_count.tail()

                      hashtag  count  count_scaled
208  harirayaganjarianspartan      1     20.111607
209            indonesiahebat      1     20.111607
210                      pdip      1     20.111607
211                  megawati      1     20.111607
212        indonesialebihbaik      1     20.111607

In [147]:
word_count.to_csv(f"{DATA_DIR}/ganjar_hashtag_count.csv", index=False)

### Archive

In [ ]:
prabowo_df[["content", "content_lemmatized"]].head()

                                              content  \
2   seseorang di snap instagram menulis kemenangan...   
7   semakin mengerikan prabowo maju sebagai presid...   
8   palak lu peang ganjar mahfud hebat prabowo ada...   
10  prabowo udah bilang rakyat yang menilai anies ...   
18  bos kamu kita pabrowo gibran memang capres yan...   

                                   content_lemmatized  
2   snap instagram menulis kemenangan prabowo huku...  
7   mengerikan prabowo maju presiden membawa misi ...  
8   palak lu peang ganjar mahfud hebat prabowo ada...  
10  prabowo bilang rakyat menilai anies suka bikin...  
18  bos pabrowo gibran capres dlm menghadapi mendu...  

In [ ]:
prabowo_df[prabowo_df.username == "@D0EMUeMnFwK7GCjncwrIhb21BSe2oPg0oLgAHxl9ohA="]["content"].values[0]

'ternyata pertanyaan paslon 01 dan 03 itu adalah pertanyaan yang tidak perlu ditanyakan kenapa coba dengar baik baik apa kata pak wiranto selaku orang yang pernah menjadi panglima abri dan pernah menjadi menhan dekade 08'

In [ ]:
prabowo_df[prabowo_df.username == "@D0EMUeMnFwK7GCjncwrIhb21BSe2oPg0oLgAHxl9ohA="]["content_lemmatized"].values[0]

'paslon 01 03 coba dengar wiranto orang panglima abri menhan dekade 08'

In [ ]:
word_count = dict()
for i in range(len(prabowo_df)):
  for word in prabowo_df.iloc[i]['content_lemmatized'].split():
      word_count[word] = word_count.get(word, 0) + 1

word_count = pd.DataFrame.from_dict(word_count, orient='index').reset_index()
word_count.columns = ['word', 'count']
word_count = word_count.sort_values(by='count', ascending=False)
word_count.head()

        word  count
4    prabowo   2081
49    gibran    604
40     anies    475
33    ganjar    326
198        2    318

In [ ]:
word_count

                word  count
4            prabowo   2081
49            gibran    604
40             anies    475
33            ganjar    326
198                2    318
...              ...    ...
5069          pakdhe      1
5070             baz      1
5071  dipersembahkan      1
1943         samakan      1
7526   fullmoonfolks      1

[7527 rows x 2 columns]

In [ ]:
# stop_words = [
#     "prabowo",
#     "gibran",
#     "orang",
#     "tak",
#     "1",
#     "subianto",
#     "mahfud",
#     "gak",
#     "kalo",
#     "3",
#     "bikin",
#     "5",
#     "4",
#     "baswedan",
#     "tdk",
#     "01",
#     "imin",
#     "pranowo",
#     "pabrowo",
#     "cak",


# ]